# Football Player Role Classification
### Comparative Analysis of a Custom CNN and Transfer Learning Models (MobileNetV2, ResNet50) with Explainable AI (Grad-CAM + SHAP)

This notebook trains and compares three deep-learning models on cropped images of people detected in football (soccer) match frames, classifying each into their **on-field role**:

- **goalkeeper**
- **player** (outfield player)
- **referee**

The source data is a **football-players-detection** dataset in YOLO format (classes: `ball`, `goalkeeper`, `player`, `referee`). The notebook reads the bounding-box annotations, **crops each annotated person**, and turns the detection dataset into a clean image-**classification** dataset organised by role. The `ball` class is excluded by default (it is an object, not a role) but can be kept via the config.

It produces **every** output required by the class activity:
- Dataset description, model-architecture tables, evaluation metrics, confusion matrices, training curves
- Grad-CAM visualisations (correct + misclassified)
- SHAP visualisations
- A final comparison table

All **figures are saved as PDF** (with PNG copies) and all **tables as CSV** into `/kaggle/working/outputs`, then the whole folder is **zipped for direct download**.

---
## Before you run (Kaggle settings)
1. **Add the dataset**: *Add Input* -> search "football players detection" and attach a YOLO-format dataset (e.g. `borhanitrash/football-players-detection-dataset` or `hamzaboulahia/football-players-detection-dataset`). The notebook auto-detects the layout, so any standard Roboflow YOLO export of this dataset works.
2. **Turn ON the GPU**: *Settings -> Accelerator -> GPU* (training is much faster).
3. **Turn ON Internet**: *Settings -> Internet -> On* (required to download the ImageNet pre-trained weights for MobileNetV2 and ResNet50). If Internet is off, the notebook still runs but those two models start from random weights and become fully trainable.
4. **Run All**.
5. When finished, download the zip from the **Output** tab: `football_player_role_outputs.zip`.


## 1. Imports, configuration and reproducibility

A single fixed seed (`42`) is used everywhere so the train/validation/test split and training are reproducible.

In [ ]:
import os, random, glob, time, json, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # safe non-interactive backend for saving figures
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2, ResNet50

from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_recall_fscore_support)

# ---- Reproducibility ----
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Configuration ----
IMG_SIZE      = (224, 224)        # all models receive 224x224x3
BATCH_SIZE    = 32
EPOCHS        = 15                 # early stopping usually stops earlier
KEEP_CLASSES  = ["goalkeeper", "player", "referee"]  # roles to classify ('ball' excluded)
MIN_BOX_PIX   = 18                 # ignore tiny crops smaller than this (px on the short side)
MAX_PER_CLASS = 1500               # cap per class to control imbalance / runtime
VAL_FRACTION  = 0.15
TEST_FRACTION = 0.15
INTERNET_ON   = True               # set False only if Kaggle Internet is off

# ---- Output folders ----
OUT_DIR   = "/kaggle/working/outputs"
FIG_DIR   = os.path.join(OUT_DIR, "figures")
TAB_DIR   = os.path.join(OUT_DIR, "tables")
for d in (OUT_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

def save_fig(fig, name):
    # Save a matplotlib figure as both PDF and PNG into the figures folder.
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=130, bbox_inches="tight")
    plt.close(fig)

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))


## 2. Dataset auto-detection

The dataset is a YOLO-format export. This cell scans `/kaggle/input` (falling back to the working directory) to locate:

- the class names (from `data.yaml` if present, otherwise a sensible default), and
- every `(image, label)` pair across the `train` / `valid` / `test` splits.

We pool all splits together and re-split ourselves later, so the notebook is robust to whatever split layout the dataset shipped with.

In [ ]:
def find_dataset_root(search_roots=("/kaggle/input", ".")):
    # Return the directory that contains a YOLO dataset (has data.yaml or labels dirs).
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            if any(f.lower() in ("data.yaml", "data.yml") for f in files):
                return dirpath
        for dirpath, dirs, _ in os.walk(root):
            if "labels" in [d.lower() for d in dirs]:
                return dirpath
    return None

def parse_class_names(root):
    # Read class names from data.yaml; fall back to the canonical football classes.
    default = ["ball", "goalkeeper", "player", "referee"]
    for cand in ("data.yaml", "data.yml"):
        p = os.path.join(root, cand)
        if os.path.exists(p):
            try:
                import yaml
                with open(p) as f:
                    data = yaml.safe_load(f)
                names = data.get("names")
                if isinstance(names, dict):   # {0: 'ball', 1: 'goalkeeper', ...}
                    names = [names[k] for k in sorted(names)]
                if names:
                    return [str(n).strip().lower() for n in names]
            except Exception as e:
                print("Could not parse data.yaml (", e, ") - using default class names.")
    print("No data.yaml found - using default class names.")
    return default

def collect_pairs(root):
    # Find all (image_path, label_path) pairs anywhere under root.
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    label_files = glob.glob(os.path.join(root, "**", "labels", "**", "*.txt"), recursive=True)
    pairs = []
    for lf in label_files:
        stem = os.path.splitext(os.path.basename(lf))[0]
        img_dir = os.path.dirname(lf).replace(os.sep + "labels", os.sep + "images")
        found = None
        for e in exts:
            cand = os.path.join(img_dir, stem + e)
            if os.path.exists(cand):
                found = cand
                break
        if found is None:  # broader search by stem
            for e in exts:
                hits = glob.glob(os.path.join(root, "**", stem + e), recursive=True)
                if hits:
                    found = hits[0]
                    break
        if found:
            pairs.append((found, lf))
    return pairs

DATA_ROOT = find_dataset_root()
assert DATA_ROOT is not None, "No YOLO dataset found under /kaggle/input. Did you attach the dataset?"
CLASS_NAMES_RAW = parse_class_names(DATA_ROOT)
PAIRS = collect_pairs(DATA_ROOT)

print("Dataset root :", DATA_ROOT)
print("Raw classes  :", CLASS_NAMES_RAW)
print("Image/label pairs found:", len(PAIRS))
assert len(PAIRS) > 0, "No (image, label) pairs found - check the dataset structure."


## 3. Build the role-classification dataset (crop the bounding boxes)

For every annotated object we crop the bounding box from its image and save it under `role_dataset/<role>/`. We:

- keep only the roles in `KEEP_CLASSES` (`goalkeeper`, `player`, `referee`),
- drop crops whose short side is below `MIN_BOX_PIX` pixels (too small to be informative),
- cap each class at `MAX_PER_CLASS` to limit the dominance of the very common `player` class.

This converts the detection dataset into a balanced, folder-per-class **image-classification** dataset.

In [ ]:
CROP_ROOT = "/kaggle/working/role_dataset"
if os.path.isdir(CROP_ROOT):
    shutil.rmtree(CROP_ROOT)
for c in KEEP_CLASSES:
    os.makedirs(os.path.join(CROP_ROOT, c), exist_ok=True)

id_to_name = {i: n for i, n in enumerate(CLASS_NAMES_RAW)}
counts = {c: 0 for c in KEEP_CLASSES}
random.shuffle(PAIRS)

for img_path, lbl_path in PAIRS:
    if all(counts[c] >= MAX_PER_CLASS for c in KEEP_CLASSES):
        break
    try:
        im = Image.open(img_path).convert("RGB")
    except Exception:
        continue
    W, H = im.size
    try:
        with open(lbl_path) as f:
            lines = [ln.strip() for ln in f if ln.strip()]
    except Exception:
        continue
    for ln in lines:
        parts = ln.split()
        if len(parts) < 5:
            continue
        cid = int(float(parts[0]))
        name = id_to_name.get(cid, "")
        if name not in KEEP_CLASSES or counts[name] >= MAX_PER_CLASS:
            continue
        cx, cy, bw, bh = map(float, parts[1:5])         # normalised YOLO box
        x1 = int((cx - bw / 2) * W); y1 = int((cy - bh / 2) * H)
        x2 = int((cx + bw / 2) * W); y2 = int((cy + bh / 2) * H)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        if (x2 - x1) < MIN_BOX_PIX or (y2 - y1) < MIN_BOX_PIX:
            continue
        crop = im.crop((x1, y1, x2, y2))
        out = os.path.join(CROP_ROOT, name, name + "_" + str(counts[name]).zfill(5) + ".jpg")
        crop.save(out, quality=90)
        counts[name] += 1

print("Cropped images per role:")
for c in KEEP_CLASSES:
    print("  ", c.ljust(12), ":", counts[c])

CLASS_NAMES = [c for c in KEEP_CLASSES if counts[c] > 0]
NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES >= 2, "Need at least two non-empty role classes."
print("Final classes used:", CLASS_NAMES)


## Figure 1 - Dataset samples

A grid of example crops for each role. This is the qualitative dataset description required by the activity.

In [ ]:
n_show = 5
fig, axes = plt.subplots(NUM_CLASSES, n_show, figsize=(n_show * 2.2, NUM_CLASSES * 2.4))
if NUM_CLASSES == 1:
    axes = np.expand_dims(axes, 0)
for r, c in enumerate(CLASS_NAMES):
    files = glob.glob(os.path.join(CROP_ROOT, c, "*.jpg"))[:n_show]
    for k in range(n_show):
        ax = axes[r][k]
        ax.axis("off")
        if k < len(files):
            ax.imshow(Image.open(files[k]).resize((96, 160)))
        if k == 0:
            ax.set_title(c, fontsize=11, loc="left")
fig.suptitle("Figure 1 - Sample crops per role", fontsize=14)
fig.tight_layout()
save_fig(fig, "Figure1_dataset_samples")

desc = pd.DataFrame({"Role": CLASS_NAMES,
                     "Num images": [counts[c] for c in CLASS_NAMES]})
desc.loc[len(desc)] = ["TOTAL", desc["Num images"].sum()]
desc["Image size"] = str(IMG_SIZE[0]) + "x" + str(IMG_SIZE[1]) + " (resized)"
desc.to_csv(os.path.join(TAB_DIR, "dataset_description.csv"), index=False)
desc


## Figure 2 - Model design workflow

A simple block diagram of the pipeline, drawn with matplotlib so it is saved alongside the other figures.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axis("off")
steps = ["Input crop\n(224x224x3)",
         "Custom CNN /\nMobileNetV2 /\nResNet50",
         "Role prediction\n(goalkeeper /\nplayer / referee)",
         "Evaluation\n(accuracy, F1,\nconfusion matrix)",
         "Explainability\n(Grad-CAM + SHAP)"]
y = 0.5
xs = np.linspace(0.08, 0.92, len(steps))
for i, (x, s) in enumerate(zip(xs, steps)):
    ax.text(x, y, s, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.5", fc="#eef4ff", ec="#3367d6"))
    if i < len(steps) - 1:
        ax.annotate("", xy=(xs[i + 1] - 0.07, y), xytext=(x + 0.07, y),
                    arrowprops=dict(arrowstyle="->", lw=1.6, color="#333"))
ax.set_title("Figure 2 - Model design workflow", fontsize=14)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
save_fig(fig, "Figure2_workflow")
print("Saved workflow diagram.")


## 4. Train / validation / test pipelines

We load the cropped images with `image_dataset_from_directory` at **raw `[0,255]` scale** (normalisation is baked into each model later, so Grad-CAM and SHAP overlays stay interpretable on the original pixels). A held-out **test** split is carved out first, then the remainder is split into **train** and **validation**. Light augmentation (flip, small rotation/zoom) is applied to the training set only.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

full_ds = tf.keras.utils.image_dataset_from_directory(
    CROP_ROOT, labels="inferred", label_mode="int",
    class_names=CLASS_NAMES, image_size=IMG_SIZE, batch_size=None,
    shuffle=True, seed=SEED)

n_total = full_ds.cardinality().numpy()
n_test  = int(n_total * TEST_FRACTION)
n_val   = int(n_total * VAL_FRACTION)
n_train = n_total - n_test - n_val
print("Total crops:", n_total, " -> train", n_train, "| val", n_val, "| test", n_test)

full_ds = full_ds.shuffle(n_total, seed=SEED, reshuffle_each_iteration=False)
test_ds  = full_ds.take(n_test)
rest_ds  = full_ds.skip(n_test)
val_ds   = rest_ds.take(n_val)
train_ds = rest_ds.skip(n_val)

y_train = np.array([int(y.numpy()) for _, y in train_ds])
cls, cnt = np.unique(y_train, return_counts=True)
class_weight = {int(c): float(len(y_train) / (len(cls) * n)) for c, n in zip(cls, cnt)}
print("Class weights:", class_weight)

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.06, seed=SEED),
    layers.RandomZoom(0.10, seed=SEED),
], name="augment")

def prep(ds, training=False):
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_pipe = prep(train_ds, training=True)
val_pipe   = prep(val_ds)
test_pipe  = prep(test_ds)


## 5. Model definitions

Three architectures. Preprocessing is **built into each model** so all three share the same raw `[0,255]` input pipeline:

- **Custom CNN** - `Rescaling(1/255)` then 3 conv blocks -> GAP -> dropout -> dense.
- **MobileNetV2** - `Rescaling(1/127.5, offset=-1)` (maps to `[-1,1]`, the scale ImageNet MobileNetV2 expects) then the frozen ImageNet base + a lightweight head.
- **ResNet50** - `resnet50.preprocess_input` then the frozen ImageNet base + a lightweight head.

For each model we also keep a reference to the **last convolutional feature tensor** at build time and build a `grad_model` from it. Capturing the tensor during construction (rather than digging it out of a nested base afterwards) is what makes Grad-CAM work cleanly on the transfer-learning models without graph-disconnect errors.

In [ ]:
def build_custom_cnn():
    inp = layers.Input(shape=IMG_SIZE + (3,))
    x = layers.Rescaling(1.0 / 255)(inp)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    feat = layers.Conv2D(128, 3, padding="same", activation="relu", name="last_conv")(x)
    x = layers.GlobalAveragePooling2D()(feat)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = models.Model(inp, out, name="Custom_CNN")
    grad_model = models.Model(inp, [feat, out])
    return model, grad_model

def build_transfer(base_ctor, preprocess_layer, name):
    inp = layers.Input(shape=IMG_SIZE + (3,))
    x = preprocess_layer(inp)
    base = base_ctor(include_top=False,
                     weights=("imagenet" if INTERNET_ON else None),
                     input_shape=IMG_SIZE + (3,))
    base.trainable = bool(not INTERNET_ON)  # frozen if we have pretrained weights
    feat = base(x)                          # last conv feature map - captured here
    x = layers.GlobalAveragePooling2D()(feat)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = models.Model(inp, out, name=name)
    grad_model = models.Model(inp, [feat, out])
    return model, grad_model

def build_mobilenet():
    pp = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mnv2_preprocess")
    return build_transfer(MobileNetV2, pp, "MobileNetV2")

def build_resnet():
    pp = layers.Lambda(tf.keras.applications.resnet50.preprocess_input,
                       name="resnet_preprocess")
    return build_transfer(ResNet50, pp, "ResNet50")

BUILDERS = {"Custom CNN": build_custom_cnn,
            "MobileNetV2": build_mobilenet,
            "ResNet50": build_resnet}

arch_rows = []
for nm, fn in BUILDERS.items():
    m, _ = fn()
    total = m.count_params()
    trainable = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    arch_rows.append({"Model": nm, "Total params": total,
                      "Trainable params": trainable,
                      "Frozen params": total - trainable})
    del m
arch_df = pd.DataFrame(arch_rows)
arch_df.to_csv(os.path.join(TAB_DIR, "model_architecture.csv"), index=False)
arch_df


## 6. Training

Each model is trained with Adam, sparse categorical cross-entropy, class weights, and **early stopping** on validation loss (`restore_best_weights=True`). We record the training history, wall-clock training time, and parameter count for each model.

In [ ]:
histories, trained, grad_models, train_times = {}, {}, {}, {}

for nm, fn in BUILDERS.items():
    print("\n=== Training", nm, "===")
    tf.random.set_seed(SEED)
    model, grad_model = fn()
    model.compile(optimizer=optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    es = callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                 restore_best_weights=True)
    t0 = time.time()
    h = model.fit(train_pipe, validation_data=val_pipe, epochs=EPOCHS,
                  class_weight=class_weight, callbacks=[es], verbose=2)
    train_times[nm] = time.time() - t0
    histories[nm] = h.history
    trained[nm] = model
    grad_models[nm] = grad_model
    print(nm, "trained in", round(train_times[nm], 1), "s")


## 7. Evaluation

For each model we compute accuracy, weighted and macro precision/recall/F1 on the held-out test set, plus a full classification report saved to CSV.

In [ ]:
X_test, y_test = [], []
for xb, yb in test_pipe:
    X_test.append(xb.numpy()); y_test.append(yb.numpy())
X_test = np.concatenate(X_test); y_test = np.concatenate(y_test)
print("Test tensor:", X_test.shape)

metric_rows, preds_store = [], {}
for nm, model in trained.items():
    probs = model.predict(X_test, verbose=0)
    yhat = probs.argmax(1)
    preds_store[nm] = yhat
    acc = accuracy_score(y_test, yhat)
    pw, rw, fw, _ = precision_recall_fscore_support(y_test, yhat, average="weighted", zero_division=0)
    pm, rm, fm, _ = precision_recall_fscore_support(y_test, yhat, average="macro", zero_division=0)
    metric_rows.append({"Model": nm, "Accuracy": round(acc, 4),
                        "Precision (w)": round(pw, 4), "Recall (w)": round(rw, 4),
                        "F1 (w)": round(fw, 4), "F1 (macro)": round(fm, 4),
                        "Train time (s)": round(train_times[nm], 1),
                        "Params": trained[nm].count_params()})
    rep = classification_report(y_test, yhat, target_names=CLASS_NAMES,
                                output_dict=True, zero_division=0)
    pd.DataFrame(rep).transpose().to_csv(
        os.path.join(TAB_DIR, "classification_report_" + nm.replace(" ", "_") + ".csv"))

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(os.path.join(TAB_DIR, "evaluation_metrics.csv"), index=False)
metrics_df


## Figure 3 - Accuracy and loss curves

Training vs validation accuracy and loss for all three models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for nm, h in histories.items():
    axes[0].plot(h["accuracy"], label=nm + " train")
    axes[0].plot(h["val_accuracy"], "--", label=nm + " val")
    axes[1].plot(h["loss"], label=nm + " train")
    axes[1].plot(h["val_loss"], "--", label=nm + " val")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
for a in axes:
    a.legend(fontsize=8); a.grid(alpha=0.3)
fig.suptitle("Figure 3 - Training and validation curves", fontsize=14)
fig.tight_layout()
save_fig(fig, "Figure3_training_curves")
print("Saved training curves.")


## Figure 4 - Confusion matrix comparison

One confusion matrix per model on the test set.

In [ ]:
fig, axes = plt.subplots(1, len(trained), figsize=(5 * len(trained), 4.4))
if len(trained) == 1:
    axes = [axes]
for ax, (nm, yhat) in zip(axes, preds_store.items()):
    cm = confusion_matrix(y_test, yhat, labels=range(NUM_CLASSES))
    ax.imshow(cm, cmap="Blues")
    ax.set_title(nm)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=9)
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        os.path.join(TAB_DIR, "confusion_matrix_" + nm.replace(" ", "_") + ".csv"))
fig.suptitle("Figure 4 - Confusion matrices", fontsize=14)
fig.tight_layout()
save_fig(fig, "Figure4_confusion_matrices")
print("Saved confusion matrices.")


## 8. Grad-CAM

Grad-CAM highlights the image regions most responsible for a prediction. We use the `grad_model` (built earlier) that returns the last convolutional feature map and the class probabilities. We show Grad-CAM for **correctly** classified images (Figure 5) and **misclassified** images (Figure 6), across all three models.

In [ ]:
def gradcam_heatmap(grad_model, img, class_idx):
    # Compute a Grad-CAM heatmap for one image (HxWx3, [0,255] scale).
    arr = tf.convert_to_tensor(img[None].astype("float32"))
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(arr)
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)[0]
    weights = tf.reduce_mean(grads, axis=(0, 1))
    cam = tf.reduce_sum(conv_out[0] * weights, axis=-1).numpy()
    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-8)
    cam = np.array(Image.fromarray((cam * 255).astype("uint8")).resize(IMG_SIZE))
    return cam / 255.0

def overlay(ax, img, cam, title):
    ax.imshow(img.astype("uint8"))
    ax.imshow(cam, cmap="jet", alpha=0.45)
    ax.set_title(title, fontsize=8); ax.axis("off")

ref_model = "ResNet50" if "ResNet50" in preds_store else list(preds_store)[0]
correct_idx   = np.where(preds_store[ref_model] == y_test)[0]
incorrect_idx = np.where(preds_store[ref_model] != y_test)[0]
rng = np.random.default_rng(SEED)
sel_correct   = rng.choice(correct_idx, size=min(3, len(correct_idx)), replace=False) if len(correct_idx) else []
sel_incorrect = rng.choice(incorrect_idx, size=min(3, len(incorrect_idx)), replace=False) if len(incorrect_idx) else []

def gradcam_grid(indices, fname, title):
    if len(indices) == 0:
        print("No samples available for", title); return
    ncol = len(trained) + 1
    fig, axes = plt.subplots(len(indices), ncol, figsize=(2.6 * ncol, 2.8 * len(indices)))
    if len(indices) == 1:
        axes = np.expand_dims(axes, 0)
    for r, idx in enumerate(indices):
        img = X_test[idx]
        axes[r][0].imshow(img.astype("uint8")); axes[r][0].axis("off")
        axes[r][0].set_title("True: " + CLASS_NAMES[y_test[idx]], fontsize=8)
        for k, nm in enumerate(trained):
            pred = preds_store[nm][idx]
            cam = gradcam_heatmap(grad_models[nm], img, pred)
            overlay(axes[r][k + 1], img, cam, nm + "\npred: " + CLASS_NAMES[pred])
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    save_fig(fig, fname)
    print("Saved", fname)

gradcam_grid(sel_correct,   "Figure5_gradcam_correct",       "Figure 5 - Grad-CAM on correct predictions")
gradcam_grid(sel_incorrect, "Figure6_gradcam_misclassified", "Figure 6 - Grad-CAM on misclassified predictions")


## 9. SHAP

SHAP estimates how each region pushes the prediction toward (red) or away from (blue) the predicted class. SHAP can be sensitive to library versions, so this cell is defensive: it tries `GradientExplainer` on a small set of test images and, if anything fails, saves a clearly labelled placeholder figure instead of stopping the notebook. The prediction function passed to SHAP works on `[0,1]` images and rescales to `[0,255]` internally, matching the models' raw-pixel input.

In [ ]:
try:
    import shap
    bg_n, ex_n = 25, 3
    rng2 = np.random.default_rng(SEED)
    bg_idx = rng2.choice(len(X_test), size=min(bg_n, len(X_test)), replace=False)
    ex_idx = rng2.choice(len(X_test), size=min(ex_n, len(X_test)), replace=False)

    # SHAP works in [0,1]; predict_fn rescales back to [0,255] for the models.
    background01 = (X_test[bg_idx] / 255.0).astype("float32")
    examples01   = (X_test[ex_idx] / 255.0).astype("float32")

    shap_model = trained[ref_model]
    def predict_fn(x01):
        return shap_model.predict((x01 * 255.0).astype("float32"), verbose=0)

    explainer = shap.GradientExplainer(shap_model, (background01 * 255.0).astype("float32"))
    shap_values = explainer.shap_values((examples01 * 255.0).astype("float32"))

    shap.image_plot(shap_values, (examples01 * 255.0).astype("float32"), show=False)
    plt.suptitle("Figure 7 - SHAP explanations (" + ref_model + ")", fontsize=12)
    plt.savefig(os.path.join(FIG_DIR, "Figure7_shap.pdf"), bbox_inches="tight")
    plt.savefig(os.path.join(FIG_DIR, "Figure7_shap.png"), dpi=130, bbox_inches="tight")
    plt.close("all")
    print("Saved SHAP figure for", ref_model)
except Exception as e:
    print("SHAP step skipped due to:", repr(e))
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axis("off")
    ax.text(0.5, 0.5,
            "SHAP could not run in this environment.\n"
            "Re-run with a compatible shap version (pip install shap).\n"
            "Reason: " + type(e).__name__,
            ha="center", va="center", fontsize=11)
    ax.set_title("Figure 7 - SHAP explanations (placeholder)")
    save_fig(fig, "Figure7_shap")


## 10. Final comparison table

The summary table requested by the activity. The **Grad-CAM Quality** and **SHAP Interpretability** columns are left blank for you to fill in (High / Medium / Low, or 1-5) after inspecting Figures 5-7.

In [ ]:
final = metrics_df[["Model", "Accuracy", "F1 (w)", "Train time (s)", "Params"]].copy()
final = final.rename(columns={"F1 (w)": "F1-Score", "Train time (s)": "Training Time"})
final["Grad-CAM Quality"] = ""
final["SHAP Interpretability"] = ""
final["Overall Finding"] = ""
final.to_csv(os.path.join(TAB_DIR, "final_comparison.csv"), index=False)
final


## 11. Package all outputs

Zip the entire `outputs/` folder (figures as PDF + PNG, all tables as CSV) for one-click download from the Kaggle **Output** tab.

In [ ]:
zip_path = "/kaggle/working/football_player_role_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for dirpath, _, files in os.walk(OUT_DIR):
        for f in files:
            fp = os.path.join(dirpath, f)
            z.write(fp, os.path.relpath(fp, OUT_DIR))
print("Wrote", zip_path)
print("\nFigures:")
for f in sorted(os.listdir(FIG_DIR)):
    print("  ", f)
print("\nTables:")
for f in sorted(os.listdir(TAB_DIR)):
    print("  ", f)


## How the outputs answer the research questions

- **RQ1 (How well does the custom CNN classify roles?)** -> Accuracy / F1 of *Custom CNN* in `evaluation_metrics.csv`.
- **RQ2 (Does transfer learning improve performance?)** -> compare MobileNetV2 and ResNet50 against the Custom CNN in `final_comparison.csv`.
- **RQ3 (Best performance/cost trade-off?)** -> weigh F1 against Training Time and Params in `final_comparison.csv`.
- **RQ4 (Where does each model look on correct predictions?)** -> Figure 5 (Grad-CAM correct). Expect attention on kit colour, goalkeeper jersey/gloves, referee uniform.
- **RQ5 (Different attention patterns, CNN vs transfer?)** -> compare columns within Figure 5.
- **RQ6 (SHAP positive/negative evidence?)** -> Figure 7 (red pushes toward the class, blue against).
- **RQ7 (Can explainability diagnose misclassifications?)** -> Figure 6 (Grad-CAM on errors) - e.g. a goalkeeper in an unusual kit confused with an outfield player.

### Filling the qualitative columns
After inspecting Figures 5-7, rate each model's **Grad-CAM Quality** and **SHAP Interpretability** as **High / Medium / Low** (or 1-5) in `final_comparison.csv`:
- *High* = highlighted regions clearly fall on the person (kit/torso) rather than the background pitch.
- *Low* = attention scattered on grass, crowd, or pitch lines.

All figures (PDF + PNG) and tables (CSV) are inside `football_player_role_outputs.zip` in the **Output** tab.